# Exercise 20 - Data Preparation and Data Augmentation

Estimated Time: **45-50 minutes**

The goals of this exercise are to gain some familiarity with the [Python Pandas](https://pandas.pydata.org/) library  and with image data augmentation in [Keras](https://keras.io). Regarding data preparation, we merely show a set of examples of using Pandas, which you can experiment with and use for reference.

- Use **T4 GPU** for this exercise

## Pandas and the California Housing dataset
First load the California Housing dataset from [scikit-learn](http://scikit-learn.org).

In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

Run the following code, which creates a Pandas DataFrame and calls `head()` to print the first 5 samples of the dataset

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

df = pd.DataFrame(housing.data, columns=housing.feature_names)
df.head()

Call `describe()` to print out common statistics such as the minimum, maximum, and mean values of each feature.

In [ ]:
df.describe()

Plot the values of all the features for the first 5 samples in the dataset.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.style.use('ggplot')
plt.figure(1, figsize=(12,8))

df.iloc[0:5].plot.bar();

Repeat for the first 10 samples, and with the features stacked (`stacked=True`) along horizonal bars (`plot.barh()`

In [ ]:
df.iloc[0:10].plot.barh(stacked=True);

Plot a set of histograms showing the distribution of values for each separate feature.

In [ ]:
df.hist(bins=50);

Plot histograms showing the distributions of three specific features on a single plot. The `alpha` parameter determines the degree of transparency of the overlaid histogram plots. Set `stacked=True` to stack the plots instead of overlaying them.

In [ ]:
df.loc[:,['MedInc','Latitude', 'Longitude']].plot.hist(alpha=0.6, bins=20); # stacked=True);

Create a box plot. The box surrounds samples in the 2nd and 3rd quartiles. The horizonal line inside the box shows the median. The end bars show the minimum and maximum values. The circles show suspected outliers.

In [ ]:
df.loc[:,['MedInc','Latitude', 'Longitude']].plot.box();

An area plot shows the values of each of the samples individually:

In [ ]:
df.loc[:,['MedInc','Latitude', 'Longitude']].plot.area(stacked=False);

A scatter plot can show values along 4 'axes': X coordinate, Y coordinate, grey scale, and circle diameter.

In [ ]:
df.plot.scatter(x='MedInc', y='Latitude', c='Longitude', s=df['HouseAge']*10);

A scatter maxtric shows a separate scatter plot for each and every possible PAIR of features:

In [ ]:
m = pd.plotting.scatter_matrix(df, figsize=(12, 12))

## Using Pandas to clean data
Now we will construct a toy dataset and show how to drop dupicates and invalid data. Data preparation may be more complicated than this in real life, but this example shows some of the main principles. The toy dataset contains 4 features named A-D.

In [ ]:
n = 5
df = pd.DataFrame(np.random.randn(n,4),index=range(1,1+n),columns=list('ABCD'))
df

Duplicate a few rows:

In [ ]:
df.loc[6] = df.loc[1]
df.loc[7] = df.loc[1]
df

Poke an invalid value into the dataset.

In [ ]:
df.loc[2, 'B'] = np.nan
df

Duplicate rows can be dropped with a single call.

In [ ]:
df = df.drop_duplicates()
df

The same goes for rows containing values that are not numbers.

In [ ]:
df = df.dropna(how='any')
df

## Image data augmentation in Keras

The goal of the final part of this exercise is to experiment with image data augmentation in Keras. This is nice for training purposes because it comes out-of-the-box with Keras.

Run the following code to load the MNIST dataset and plot a few samples.

In [ ]:
%matplotlib inline
from tensorflow.keras.datasets import mnist
import matplotlib
import matplotlib.pyplot as plt

# The data, shuffled and split between train and test sets:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Plot some images as a sanity check
matplotlib.style.use('seaborn-v0_8-white')
plt.figure(1, figsize=(5,5))
for i in range(0, 9):
    plt.subplot(330 + 1 + i)
    plt.imshow(x_train[i], cmap=plt.get_cmap('gray'))
plt.show()

Run each of the code fragments below, which demonstrate several of the possible image transformations provided by the [Keras `ImageDataGenerator`](https://keras.io/preprocessing/image/).

In [ ]:
import tensorflow
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def load_mnist_and_reshape():
    global x_train, y_train, x_test, y_test
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    x_train = x_train.reshape(-1, 28, 28, 1)
    x_test  = x_test.reshape(-1, 28, 28, 1)

def generate_and_plot_a_few_images(n = 9):
    x, y = next(datagen.flow(x_train, y_train, batch_size=n))
    for i in range(0, n):
        plt.subplot(330 + 1 + i)
        plt.imshow(x[i].reshape(28, 28), cmap=plt.get_cmap('gray'))
    plt.show()

In [ ]:
# Random Shifts
load_mnist_and_reshape()
shift = 0.3
datagen = ImageDataGenerator(width_shift_range=shift, height_shift_range=shift)
datagen.fit(x_train)
generate_and_plot_a_few_images()

In [ ]:
# Random Flips
load_mnist_and_reshape()
datagen = ImageDataGenerator(horizontal_flip=True, vertical_flip=True)
datagen.fit(x_train)
generate_and_plot_a_few_images()

In [ ]:
# Random Shear
load_mnist_and_reshape()
datagen = ImageDataGenerator(shear_range=1.)
datagen.fit(x_train)
generate_and_plot_a_few_images()

In [ ]:
# Random zoom
load_mnist_and_reshape()
datagen = ImageDataGenerator(zoom_range=[.8, 2.5])
datagen.fit(x_train)
generate_and_plot_a_few_images()

Now we're ready to try training a network using augmented image data. First run the code below to prepare the MNIST dataset and an augmented test dataset.

In [ ]:
from tensorflow.keras.utils import to_categorical

image_size = 28
n_labels   = 10

load_mnist_and_reshape()

# Convert the labels 0-9 to one-hot
y_train = to_categorical(y_train, n_labels)
y_test  = to_categorical(y_test, n_labels)

print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)

# Generate an augmented test dataset
datagen = ImageDataGenerator(rotation_range=30, width_shift_range=0.1, height_shift_range=0.1, shear_range=.5,
                             zoom_range=[.9, 1.1], horizontal_flip=False, vertical_flip=False )

x_augmented, y_augmented = next(datagen.flow(x_test, y_test, batch_size=x_test.shape[0]))
print('x_augmented.shape:', x_augmented.shape)

# Plot a few images from the augmented test dataset
for i in range(0, 9):
    plt.subplot(330 + 1 + i)
    plt.imshow(x_augmented[i].reshape(28, 28), cmap=plt.get_cmap('gray'))
plt.show()

Build and run a Keras model using the original MNIST training dataset as a benchmark.

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D

patch_size     =   3
pool_size      =   2
n_features     =  30
n_full_units   = 128
batch_size     = 100

def build_keras_model():
    tensorflow.keras.backend.clear_session()

    model = Sequential()
    model.add(Input(shape=(image_size,image_size,1)))
    model.add(Conv2D(n_features, (patch_size,patch_size), padding='same', activation='relu'))
    model.add(MaxPooling2D(pool_size=(pool_size,pool_size)))
    model.add(Conv2D(n_features, (patch_size,patch_size), padding='same', activation='relu'))
    model.add(MaxPooling2D(pool_size=(pool_size,pool_size)))
    model.add(Flatten())
    model.add(Dense(units=n_full_units, activation='relu'))
    model.add(Dense(units=n_full_units, activation='relu'))
    model.add(Dense(units=n_labels, activation='softmax'))
    #model.summary()

    return model

Train the model on the MNIST training dataset and then evaluate on both the MNIST test dataset and the augmented test dataset.

In [ ]:
model = build_keras_model()
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=batch_size, shuffle=True)

loss_and_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)
print(f'\nAccuracy on MNIST test dataset     = {loss_and_acc[1]:4.2f}')

loss_and_acc = model.evaluate(x_augmented, y_augmented, batch_size=batch_size, verbose=0)
print(f'Accuracy on augmented test dataset = {loss_and_acc[1]:4.2f}')

As expected, the model quickly achieves a high accuracy on the MNIST test dataset (MNIST is an easy problem) but a considerably lower accuracy on the augmented test dataset that the model has not seen before.

Now complete the code below to build, compile, and train the model on an augmented training dataset (see the [documentation](https://keras.io/preprocessing/image/)). Evaluate the model on both the MNIST test dataset and the augmented test dataset, as was done immediately above. You will need to train the model for longer than 5 epochs to get good results.

**Beware using any of the featurewise transformations** (featurewise_center, featurewise_std_normalization, zca_whitening) because the fit() method of class ImageDataGenerator seems to hang in recent versions of Keras when using these image transformations! (It used to work in earlier versions!)

In [ ]:
#

#### Solution

Here is our answer. Do not run the cell below unless you want to see the answer we provide

<details>
    <summary> Click here for the answer </summary>
    
    model = build_keras_model()
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(datagen.flow(x_train, y_train, batch_size), epochs=10, steps_per_epoch=x_train.shape[0] // batch_size)

    loss_and_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)
    print(f'\nAccuracy on MNIST test dataset     = {loss_and_acc[1]:4.2f}')

    loss_and_acc = model.evaluate(x_augmented, y_augmented, batch_size=batch_size, verbose=0)
    print(f'Accuracy on augmented test dataset = {loss_and_acc[1]:4.2f}')
    
</details>